# 1. Simple RAG

The minimal RAG loop: chunk, embed, retrieve, generate. Start here.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Large language models only know what was in their training data, they cannot see your private documents, and they confidently invent facts when they don't know. **Retrieval-Augmented Generation (RAG)** fixes this by looking relevant passages up at question time and putting them in the prompt. This notebook builds the smallest complete version so every later technique has a baseline to beat.

## How it works

1. **Load** the source document
2. **Chunk** it into overlapping pieces small enough to embed precisely
3. **Embed** every chunk into a vector with a sentence-transformer
4. **Index** the vectors in a store
5. **Retrieve** the chunks whose vectors are closest to the question's vector
6. **Generate** an answer from only those chunks

```text
 documents --> [chunk] --> [embed] --> vector store
                                           |
 question ---> [embed] --> similarity search --> top-k chunks --> [LLM + prompt] --> answer
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 10:23:42.097000 6674 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


## 1. Retrieve
Embed the question and find the closest chunks by cosine similarity.

In [2]:
query = "How many dimensions does the all-MiniLM-L6-v2 model produce?"
hits = store.search(query, k=3)
show(hits)

0.431  has four stages: ingest documents, split them into chunks, index the chunks as embeddings, and at query time retrieve the closest chunks and generate an answer ...
0.205  r and outputs a relevance score, which is far more accurate than comparing two independent vectors. Because cross-encoders are slow, they are applied only to th...
0.169  cuments and falls back to web search when they are irrelevant. Self-RAG trains the model to emit reflection tokens that decide whether retrieval is needed and w...


## 2. Generate
Pass the retrieved chunks to the LLM as context.

In [3]:
response, hits = rag(query, store, k=3)
print(response)

According to [1], the all-MiniLM-L6-v2 model produces 384-dimensional vectors.


## 3. Try it yourself
Ask something the handbook does not cover. A grounded system should say it doesn't know.

In [4]:
print(rag("Who won the 2018 football world cup?", store)[0])

I don't know. The context does not provide information about how to answer questions like "Who won the 2018 football world cup?" or what retrieval system would be most effective for such a question. It only discusses general concepts related to search systems, embeddings, and ranking functions.


## Look inside the pipeline
Before trusting the black box, inspect each stage. First, what do the chunks look like and how big are they?

In [5]:
import pandas as pd
sizes = pd.Series([len(c) for c in chunks])
print(sizes.describe().round(1))
print("\nFirst chunk:\n", chunks[0])
print("\nSecond chunk (note the overlap with the first):\n", chunks[1])

count     19.0
mean     476.2
std       84.1
min      129.0
25%      494.0
50%      495.0
75%      497.5
max      499.0
dtype: float64

First chunk:
 # The Applied AI Handbook

## Introduction to RAG

Retrieval-Augmented Generation (RAG) combines a retriever with a language model. Instead of relying only on what the model memorised during training, the system first searches an external knowledge base and then passes the retrieved passages to the model as context. This keeps answers grounded in up-to-date, private or domain-specific data without retraining the model. A basic RAG pipeline has four stages: ingest documents, split them into

Second chunk (note the overlap with the first):
 has four stages: ingest documents, split them into chunks, index the chunks as embeddings, and at query time retrieve the closest chunks and generate an answer from them.

## Embeddings

An embedding model maps text to a dense vector so that texts with similar meaning land close together. The all-MiniLM-

### What does an embedding look like?
Each chunk becomes a 384-number vector. The numbers are meaningless individually - only distances between vectors matter.

In [6]:
v = store.vecs[0]
print("shape of the index:", store.vecs.shape, "(chunks x dimensions)")
print("first 8 numbers of chunk 0:", v[:8].round(3))
print("vector length (should be 1.0 because we normalise):", float((v ** 2).sum() ** 0.5))

shape of the index: (19, 384) (chunks x dimensions)
first 8 numbers of chunk 0: [-0.076  0.044  0.032  0.037 -0.043  0.047 -0.012 -0.019]
vector length (should be 1.0 because we normalise): 1.0000001192092896


### Similarity in action
Cosine similarity of unit vectors is just a dot product. Related sentences score high, unrelated ones low.

In [7]:
pairs = [
    ("How do I search vectors quickly?", "HNSW and IVF are approximate nearest neighbour indexes."),
    ("How do I search vectors quickly?", "Marie Curie discovered polonium."),
    ("How is similarity between vectors measured?", "Cosine similarity compares the direction of two vectors."),
]
for a, b in pairs:
    print(f"{float(embed(a)[0] @ embed(b)[0]):.3f}  {a!r} <-> {b!r}")

0.289  'How do I search vectors quickly?' <-> 'HNSW and IVF are approximate nearest neighbour indexes.'
0.093  'How do I search vectors quickly?' <-> 'Marie Curie discovered polonium.'
0.721  'How is similarity between vectors measured?' <-> 'Cosine similarity compares the direction of two vectors.'


### Anatomy of the prompt
This is exactly what the LLM receives. Notice the instruction to stay inside the context - that single line is what turns a chatty model into a grounded one.

In [8]:
q = "What is HNSW?"
hits = store.search(q, k=2)
context = "\n\n".join(f"[{i + 1}] {h['text']}" for i, h in enumerate(hits))
print("SYSTEM:", RAG_SYSTEM, "\n")
print(f"USER:\nContext:\n{context}\n\nQuestion: {q}")

SYSTEM: Answer the question using only the provided context. If the context does not contain the answer, say you don't know. 

USER:
Context:
[1] res the query with every vector, which is accurate but slow at scale. Approximate nearest neighbour indexes such as HNSW and IVF trade a little recall for large speed gains. HNSW builds a layered graph where upper layers provide long jumps and lower layers refine the result. FAISS is a library for in-process vector search, while Milvus, Qdrant and Weaviate run as separate database services with filtering, persistence and replication.

## Chunking Strategies

Documents must be split before

[2] ten vague, short or badly phrased, so the query is rewritten before retrieval. Query rewriting turns a conversational question into a standalone search query. Step-back prompting asks a more general question first to fetch broader background. Sub-query decomposition splits a complex question into several simple ones that are retrieved separately. HyDE, 

### Effect of `k` (how many chunks to retrieve)
Too few and the answer may be missing; too many and noise or cost goes up. Measure it on the golden questions rather than guessing.

In [9]:
import pandas as pd
rows = []
for k in [1, 2, 3, 5, 8]:
    r = evaluate_retriever(store.search, k=k)
    rows.append({"k": k, "hit_rate@1": r["hit_rate@1"], "hit_rate@k": r[f"hit_rate@{k}"], "MRR": r["MRR"]})
pd.DataFrame(rows).set_index("k")

,hit_rate@1,hit_rate@k,MRR
k,,,
1,0.8,0.80,0.80
2,0.8,0.95,0.88
3,0.8,1.00,0.89
5,0.8,1.00,0.89
8,0.8,1.00,0.89


### Failure case: asking about something not in the documents
A grounded system must refuse rather than guess.

In [10]:
for q in ["What is the capital of Australia?", "What is the warranty period of the AeroBook laptop?"]:
    print("Q:", q, "\nA:", rag(q, store, k=3)[0], "\n")

Q: What is the capital of Australia? 
A: I don't know. The context does not provide information about how to answer this specific question or what system would be used to retrieve the answer. 

Q: What is the warranty period of the AeroBook laptop? 
A: I don't know. The provided context doesn't mention anything about laptops or their warranties, nor does it provide information on how to answer questions like this. 



## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Tiny, fast, easy to debug | Single-shot retrieval: vague or multi-part questions often fail |
| Works with any LLM and any document type once it is text | Fixed-size chunks can cut ideas in half |
| No training required; update knowledge by re-indexing | No check that the answer is actually supported |

## When to use it

Always start here. Keep this notebook's pipeline as the baseline and only adopt a more complex technique when it beats this one on **your** questions.

## Experiments to try

- Change `size` and `overlap` in the setup cell and re-run - which questions break first?
- Swap the embedding model by setting `RAG_EMBED=all-mpnet-base-v2` and compare hit rates.
- Remove the 'only use the context' instruction from `RAG_SYSTEM` and ask an out-of-scope question.

## Key takeaways

- RAG = retrieve relevant text, then generate from it.
- Embeddings turn meaning into geometry; cosine similarity finds neighbours.
- The prompt must forbid guessing, otherwise retrieval doesn't help with hallucinations.
- Measure retrieval (hit rate) separately from generation.

---
**Next:** [2. Simple CSV RAG](../02_simple_csv_rag/simple_csv_rag.ipynb)